# coaxial 63100 - two sessions, one board
Checked in with the stand-in's outputs; `SIMULATED = True                    # False at the bench` and a
port at the bench.
`SIMULATED = True` needs no cable; every value is then invented.

The board is one slave on one wire, so two masters split a frame. A broker
owns the port and forwards Modbus requests unchanged - unit, function,
payload - and serialises them. What it removes is the exclusive OWNERSHIP,
not the exclusivity of the wire.

NOBODY STARTS IT. The device session spawns one for the port it found, and
it takes itself down when its last client goes. This example does not
mention it until it asks how many are attached.

In [1]:
import os
import subprocess
import sys
import time

from pathlib import Path

root = Path.cwd()
while not (root / 'host' / 'coaxial').is_dir():
    root = root.parent
sys.path.insert(0, str(root / 'host'))

from coaxial import Coaxial63100, broker

SIMULATED = True                    # False at the bench
PORT = 'COM4'

device = Coaxial63100(port=PORT, simulated_device=SIMULATED)
device.open()
print(device)

<Coaxial63100 Simulated SIMULATED>


## Who else is here
`broker.clients()` counts the sessions using the port. Asking is not using:
this attaches and closes, and does not count itself - otherwise the last
one out would be whoever wondered whether anybody was in.

In [2]:
print('serving:', (broker.serving() or {}).get('serial', 'nothing'))
print('sessions:', broker.clients())

serving: nothing


sessions: None


## A second session, in its own process
It opens the same port. Without the broker this is `could not open port`;
with it, both talk. Nothing here says which - that is the point.

In [3]:
SECOND = '''
import sys
sys.path.insert(0, %r)
from coaxial import Coaxial63100
with Coaxial63100(port=%r) as device:
    print('   second session sees firmware', device.system.version()['firmware'])
    import time
    time.sleep(6)
''' % (sys.path[0], PORT)

other = subprocess.Popen([sys.executable, '-c', SECOND])
time.sleep(3)
print('sessions now:', broker.clients())

sessions now: None


## Both read the same board
Whatever one session does to the board, the other sees. A reading that
moved because somebody else armed the stage should not be a mystery, which
is why the views put the count in their banner.

In [4]:
print('device  session sees firmware',
      device.system.version()['firmware'])
print('and a dead time of %d ns' % device.gates.state()['deadtime_ns'])

other.wait()
time.sleep(1)
print('sessions after it left:', broker.clients())

device  session sees firmware simulated
and a dead time of 79 ns


sessions after it left: None


## Leaving
The last one out takes the broker down, so the port is free for anything
that wants it raw - the conformance suite sends deliberately malformed
frames, which is the one thing a broker cannot forward.

In [5]:
device.close()
time.sleep(1)
print('serving after the last session left:',
      (broker.serving() or {}).get('serial', 'nothing'))

serving after the last session left: nothing
